# Эксперименты и цена усложнения
Notebook читает результаты реально выполненных запусков. Обучение и оценка реализованы в `src`, повторные команды — в README. Test здесь не используется для выбора модели.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = ROOT.parent

In [2]:
experiments = pd.read_csv(ROOT / "reports/experiments.csv")
experiments[["name", "macro_f1", "train_seconds", "single_ms", "model_bytes"]].sort_values(
    "macro_f1", ascending=False
)

,name,macro_f1,train_seconds,single_ms,model_bytes
2,svm_c0.5,0.851928,0.556643,1.0315,1891971
4,svm_c2,0.851518,0.683656,1.0180,1505145
3,svm_c1,0.850016,0.612101,0.9759,1654701
14,lr_char,0.840350,6.453828,3.8747,4472765
8,lr_c2,0.834686,4.198874,2.7991,4200561
13,lr_unigram,0.829420,1.017604,1.2992,724970
12,lr_lemma,0.820634,7.509267,2.9327,4100333
16,lr_balanced,0.818345,2.725639,2.8360,4200094
10,lr_punct,0.809704,3.667877,2.7849,4199984
1,lr_base,0.809704,3.654719,3.1433,4199951


In [3]:
transformer = json.loads((ROOT / "reports/transformer.json").read_text())
{k: v for k, v in transformer.items() if k not in ["history"]}

{'name': 'distilbert_2epochs',
 'model_id': 'distilbert/distilbert-base-uncased',
 'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be',
 'config': {'method': 'transformer', 'epochs': 2},
 'license': 'Apache-2.0',
 'device': 'cuda',
 'gpu': 'NVIDIA GeForce RTX 4050 Laptop GPU',
 'epochs': 2,
 'seed': 2026,
 'lr': 5e-05,
 'batch_size': 16,
 'max_length': 96,
 'train_seconds': 80.89846699999907,
 'validation': {'accuracy': 0.875441250630358,
  'macro_f1': 0.8681220022032152,
  'weighted_f1': 0.874587284656971,
  'top2_accuracy': 0.9435199193141705,
  'macro_ap': 0.9464677638084941},
 'batch_ms_per_item': 0.7383484619272495,
 'single_ms': 8.264799998869421,
 'model_bytes': 269013758,
 'peak_gpu_mb': 1301.076171875,
 'torch': '2.8.0+cu128'}

In [4]:
selection = json.loads((ROOT / "reports/selection.json").read_text())
{k: selection[k] for k in ["research_model", "practical_model", "temperature", "policy"]}

{'research_model': 'distilbert_2epochs',
 'practical_model': 'distilbert_2epochs',
 'temperature': 0.7655437727618911,
 'policy': {'threshold': 0.75,
  'accepted': 1627,
  'coverage': 0.8204740292486132,
  'accuracy': 0.9508297480024586,
  'human': 356}}

In [5]:
cal = json.loads((ROOT / "reports/calibration.json").read_text())
pd.DataFrame({k: {m: cal[k][m] for m in ["nll", "brier", "ece"]} for k in ["before", "after"]}).T

,nll,brier,ece
before,0.489285,0.191592,0.085405
after,0.432499,0.182123,0.017696


![Calibration](../reports/figures/calibration.png)

Сначала выбор по validation, затем заморозка артефакта и один финальный запуск оценки test. Малые отличия одного seed не доказывают превосходства метода.

In [6]:
examples = pd.read_csv(ROOT / "reports/prediction-examples.csv")
examples[examples["sample"] == "error"].head(10)

,text,true,predicted,confidence,correct,runner_up,sample
10,Does it cost me to add cash?,top_up_by_bank_transfer_charge,top_up_by_card_charge,0.747111,False,top_up_by_cash_or_cheque,error
11,I have a friend who needs money as soon as pos...,transfer_not_received_by_recipient,transfer_timing,0.827496,False,pending_transfer,error
12,I hope you can help me. My account has been co...,cash_withdrawal_not_recognised,compromised_card,0.187013,False,beneficiary_not_allowed,error
13,My top up did not complete.,pending_top_up,top_up_failed,0.693314,False,pending_top_up,error
14,my identity hasn't been verified and i can't e...,edit_personal_details,unable_to_verify_identity,0.555462,False,why_verify_identity,error
15,You can use it anywhere that accepts Mastercard.,card_acceptance,visa_or_mastercard,0.970060,False,supported_cards_and_currencies,error
16,Where is my card?,card_arrival,order_physical_card,0.355514,False,lost_or_stolen_card,error
17,WHAT IS THE SOLUTION OF THIS PROBLEM,card_arrival,verify_my_identity,0.347173,False,why_verify_identity,error
18,"hey, I think someone stole my card numbers. Th...",direct_debit_payment_not_recognised,compromised_card,0.532808,False,card_payment_not_recognised,error
19,How do I replace my card?,card_about_to_expire,order_physical_card,0.320501,False,card_linking,error


In [7]:
final = json.loads((ROOT / "reports/test-results.json").read_text())
final["practical"]

{'model': 'distilbert_2epochs',
 'official': {'accuracy': 0.8831168831168831,
  'macro_f1': 0.8819323034078578,
  'weighted_f1': 0.881932303407858,
  'top2_accuracy': 0.9470779220779221,
  'macro_ap': 0.9458161375769492},
 'decontaminated': {'accuracy': 0.8746774788057501,
  'macro_f1': 0.8725936375861522,
  'weighted_f1': 0.8739510464329119,
  'top2_accuracy': 0.9413932915591596,
  'macro_ap': 0.9396682468109248},
 'calibration': {'nll': 0.42522597851084665,
  'brier': 0.1746662144756503,
  'ece': 0.011332829499786564,
  'bins': [{'count': 7,
    'confidence': 0.16941049695014954,
    'accuracy': 0.42857142857142855},
   {'count': 19,
    'confidence': 0.266500324010849,
    'accuracy': 0.2631578947368421},
   {'count': 51,
    'confidence': 0.35756686329841614,
    'accuracy': 0.47058823529411764},
   {'count': 116,
    'confidence': 0.4491358697414398,
    'accuracy': 0.4482758620689655},
   {'count': 156,
    'confidence': 0.5494884848594666,
    'accuracy': 0.5833333333333334},
  

Ручной разбор: [error-analysis](../reports/error-analysis.md). Интерпретация, ограничения и выводы: [технический отчёт](../reports/technical-report.md).